In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim.models import FastText
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations
import random

In [2]:
EMB_DIR = "embeddings"

groups = ["entry", "mid", "senior", "leadership"]

models = {
    g: FastText.load(os.path.join(EMB_DIR, f"fasttext_{g}.model"))
    for g in groups
}

print(models.keys())

dict_keys(['entry', 'mid', 'senior', 'leadership'])


In [3]:
male_terms = [
    "he", "him", "his", "man", "male", "men", "boy", "boys",
    "father", "brother", "son", "uncle", "husband", "dad", "guy",
    "gentleman", "gentlemen", "king", "prince", "sir", "mr"
]

female_terms = [
    "she", "her", "hers", "woman", "female", "women", "girl", "girls",
    "mother", "sister", "daughter", "aunt", "wife", "mom", "lady",
    "ladies", "queen", "princess", "mrs", "ms"
]

career_terms = [
    "career", "professional", "profession", "occupation", "job", "work",
    "office", "corporate", "industry", "business", "enterprise",
    "management", "manager", "leadership", "leader", "executive",
    "director", "authority", "command", "supervisor", "boss",
    "promotion", "salary", "income", "achievement", "success",
    "ambition", "competitive", "productivity", "responsibility",
    "strategy", "decision", "innovation", "performance"
]

family_terms = [
    "family", "home", "household", "children", "child", "kids",
    "parent", "parents", "motherhood", "fatherhood",
    "marriage", "spouse", "relatives", "kin", "care",
    "support", "nurture", "nurturing", "caregiving",
    "domestic", "housework", "housekeeping",
    "dependents", "familycare"
]

In [4]:
def filter_vocab(words, model):
    return [w for w in words if w in model.wv.key_to_index]

def cosine_sim(model, w1, w2):
    v1 = model.wv[w1].reshape(1, -1)
    v2 = model.wv[w2].reshape(1, -1)
    return cosine_similarity(v1, v2)[0][0]

def association(model, w, A, B):
    return np.mean([cosine_sim(model, w, a) for a in A]) - np.mean(
        [cosine_sim(model, w, b) for b in B]
    )

def weat_effect_size(model, X, Y, A, B):
    s_X = np.array([association(model, x, A, B) for x in X])
    s_Y = np.array([association(model, y, A, B) for y in Y])

    numerator = np.mean(s_X) - np.mean(s_Y)
    denominator = np.std(np.concatenate([s_X, s_Y]), ddof=1)

    if denominator == 0:
        return np.nan

    return numerator / denominator

def weat_test_statistic(model, X, Y, A, B):
    return np.sum([association(model, x, A, B) for x in X]) - np.sum(
        [association(model, y, A, B) for y in Y]
    )

In [5]:
#permutation test
def permutation_p_value(model, X, Y, A, B, num_samples=1000, seed=42):
    random.seed(seed)
    observed = weat_test_statistic(model, X, Y, A, B)

    combined = X + Y
    size_x = len(X)

    count = 0
    for _ in range(num_samples):
        shuffled = combined[:]
        random.shuffle(shuffled)
        X_i = shuffled[:size_x]
        Y_i = shuffled[size_x:]
        stat_i = weat_test_statistic(model, X_i, Y_i, A, B)
        if stat_i >= observed:
            count += 1

    return (count + 1) / (num_samples + 1)

In [6]:
for group, model in models.items():
    X = filter_vocab(male_terms, model)
    Y = filter_vocab(female_terms, model)
    A = filter_vocab(career_terms, model)
    B = filter_vocab(family_terms, model)

    print("\n" + "=" * 70)
    print("GROUP:", group)
    print("Male terms in true vocab:", X)
    print("Female terms in true vocab:", Y)
    print("Career terms in true vocab:", A)
    print("Family terms in true vocab:", B)
    print(
        "Counts ->",
        f"male={len(X)}, female={len(Y)}, career={len(A)}, family={len(B)}"
    )


GROUP: entry
Male terms in true vocab: []
Female terms in true vocab: []
Career terms in true vocab: ['work', 'office', 'industry', 'business', 'enterprise', 'management', 'manager', 'leadership', 'director', 'strategy', 'performance']
Family terms in true vocab: ['support']
Counts -> male=0, female=0, career=11, family=1

GROUP: mid
Male terms in true vocab: []
Female terms in true vocab: []
Career terms in true vocab: ['work', 'office', 'industry', 'business', 'management', 'leadership', 'director', 'performance']
Family terms in true vocab: ['support']
Counts -> male=0, female=0, career=8, family=1

GROUP: senior
Male terms in true vocab: []
Female terms in true vocab: []
Career terms in true vocab: ['work', 'office', 'industry', 'business', 'management', 'leadership', 'director', 'performance']
Family terms in true vocab: ['support']
Counts -> male=0, female=0, career=8, family=1

GROUP: leadership
Male terms in true vocab: []
Female terms in true vocab: []
Career terms in true vo

In [ ]:
results = []

for group, model in models.items():
    X = filter_vocab(male_terms, model)
    Y = filter_vocab(female_terms, model)
    A = filter_vocab(career_terms, model)
    B = filter_vocab(family_terms, model)

    print("\n" + "=" * 60)
    print("GROUP:", group)
    print("Male terms used:", X)
    print("Female terms used:", Y)
    print("Career terms used:", A)
    print("Family terms used:", B)
    print("Counts:", len(X), len(Y), len(A), len(B))

    if min(len(X), len(Y), len(A), len(B)) < 3:
        print("Skipping due to insufficient vocab coverage")
        results.append({
            "group": group,
            "effect_size": np.nan,
            "p_value": np.nan,
            "n_male": len(X),
            "n_female": len(Y),
            "n_career": len(A),
            "n_family": len(B),
        })
        continue

    print("Computing effect size...")
    effect = weat_effect_size(model, X, Y, A, B)

    print("Computing permutation p-value...")
    p_val = permutation_p_value(model, X, Y, A, B, num_samples=100)

    print("Done:", group, effect, p_val)

    results.append({
        "group": group,
        "effect_size": effect,
        "p_value": p_val,
        "n_male": len(X),
        "n_female": len(Y),
        "n_career": len(A),
        "n_family": len(B),
    })

weat_df = pd.DataFrame(results)
weat_df

In [ ]:
weat_df.to_csv("weat_results_by_seniority.csv", index=False)
print("Saved -> weat_results_by_seniority.csv")

In [ ]:
plot_df = weat_df.dropna(subset=["effect_size"]).copy()

plt.figure(figsize=(8, 4))
plt.plot(plot_df["group"], plot_df["effect_size"], marker="o")
plt.title("WEAT Effect Size by Seniority Level")
plt.xlabel("Seniority Group")
plt.ylabel("WEAT Effect Size")
plt.tight_layout()
plt.show()

In [ ]:
# Plot p-values
plot_df = weat_df.dropna(subset=["p_value"]).copy()

plt.figure(figsize=(8, 4))
plt.plot(plot_df["group"], plot_df["p_value"], marker="o")
plt.axhline(0.05, linestyle="--")
plt.title("WEAT P-Value by Seniority Level")
plt.xlabel("Seniority Group")
plt.ylabel("P-Value")
plt.tight_layout()
plt.show()

In [ ]:
def interpret_effect(effect, p_value):
    if pd.isna(effect) or pd.isna(p_value):
        return "Insufficient vocabulary coverage."

    direction = (
        "male-career association"
        if effect > 0
        else "female-career association"
        if effect < 0
        else "no directional association"
    )

    significance = (
        "statistically significant"
        if p_value < 0.05
        else "not statistically significant"
    )

    return f"{direction}; {significance}"

weat_df["interpretation"] = weat_df.apply(
    lambda row: interpret_effect(row["effect_size"], row["p_value"]),
    axis=1
)

weat_df